# Mapping: the wall-to-wall old-growth product
-------

The selected model (XGBoost + TESSERA), the best within the AOI, is tuned once by
cross-validation and trained on all labelled parcels, then applied to every valid pixel of the
parcel-map area. This notebook displays the final hyperparameters, the wall-to-wall probability and
binary maps, the calibrator comparison (only the selected calibrated probability is stored
in the parcel product), and the predicted old-growth area with its block-retraining
bootstrap interval against the four existing products. It reads `results/mapping/<latest>/`
and renders whatever is complete; run `scripts/run_final_inference.py` and
`scripts/run_area_bootstrap.py` to populate it.

In [ ]:
NOTEBOOK = "010_mapping"

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.paths import get_project_paths
from utils.style import get_figure_size, save_figure, use_publication_style
from utils.terminology import COMPARISON_STUDIES, FOLD_IDS, PALETTE_CATEGORICAL

use_publication_style()
plt.rcParams.update(
    {
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
    }
)

paths = get_project_paths()
mapping_root = paths.results / "mapping"
runs = (
    sorted((p for p in mapping_root.iterdir() if p.is_dir()), reverse=True)
    if mapping_root.is_dir()
    else []
)
run = next((p for p in runs if (p / "run_metadata.json").is_file()), None)


def _load_oof(source_run, template):
    """Concatenate the per-fold out-of-fold predictions, or None if absent."""
    files = [source_run / template.format(fold=fold) for fold in FOLD_IDS]
    present = [f for f in files if f.is_file()]
    return pd.concat([pd.read_parquet(f) for f in present], ignore_index=True) if present else None


if run is None:
    print("No mapping run yet; run scripts/run_final_inference.py.")
    metadata = best_params = None
    calibration = calibration_pixel = area_boot = product_areas = None
    oof_parcels = oof_pixels = None
else:
    metadata = json.loads((run / "run_metadata.json").read_text())
    best_params = json.loads((run / "final_best_params.json").read_text())
    cal_path = run / "calibration_comparison.csv"
    calibration = pd.read_csv(cal_path) if cal_path.is_file() else None
    cal_pixel_path = run / "calibration_comparison_pixel.csv"
    calibration_pixel = pd.read_csv(cal_pixel_path) if cal_pixel_path.is_file() else None
    boot_path = run / "area_bootstrap.csv"
    area_boot = pd.read_csv(boot_path) if boot_path.is_file() else None
    areas_path = paths.repo_root / "data/processed/rasters/existing_products_10m/product_areas.csv"
    product_areas = pd.read_csv(areas_path) if areas_path.is_file() else None
    # The parcel- and pixel-level out-of-fold predictions used to fit and select the calibrators
    # live in the source nested-CV run; reload both to draw the reliability curves below.
    source_run = paths.repo_root / metadata["source_run"] if metadata.get("source_run") else None
    oof_parcels = (
        _load_oof(source_run, "parcel_predictions_fold{fold}.parquet") if source_run else None
    )
    oof_pixels = _load_oof(source_run, "predictions_fold{fold}.parquet") if source_run else None
    n_par = 0 if oof_parcels is None else len(oof_parcels)
    n_pix = 0 if oof_pixels is None else len(oof_pixels)
    n_boot = 0 if area_boot is None else len(area_boot)
    print(f"[load] mapping run {run.name}")
    print(
        f"[load] selected calibrator: parcel {metadata.get('selected_calibrator')}, "
        f"pixel {metadata.get('selected_calibrator_pixel')}"
    )
    parcel_area = metadata.get("predicted_ogf_area_parcel_ha")
    print(
        f"[load] predicted area: pixel {metadata.get('predicted_ogf_area_ha'):.0f} ha"
        + (f", parcel-consistent {parcel_area:.0f} ha" if parcel_area is not None else "")
    )
    print(f"[load] area-bootstrap replicates complete: {n_boot}")
    print(f"[load] OOF for reliability curves: parcels {n_par}, pixels {n_pix}")

## Final model and wall-to-wall maps

In [ ]:
if metadata is not None:
    print("Final hyperparameters (CV-tuned single search):")
    print(pd.Series(best_params).to_string())
    print(f"\nCV-tuned pooled parcel PR-AUC: {metadata.get('cv_tuned_pooled_parcel_pr_auc'):.4f}")
    print(f"Pixel operating threshold (uncalibrated F1-max): {metadata.get('pixel_threshold'):.3f}")
    print(f"Parcel-map pixels mapped: {metadata.get('n_mapped_pixels'):,}")

In [ ]:
import rasterio

if run is not None and (run / "ogf_probability_3035_10m.tif").is_file():
    fig, (ax_p, ax_b) = plt.subplots(
        1, 2, figsize=get_figure_size("double", aspect=0.5), constrained_layout=True
    )
    with rasterio.open(run / "ogf_probability_3035_10m.tif") as src:
        prob = src.read(1, masked=True)
    with rasterio.open(run / "ogf_binary_3035_10m.tif") as src:
        binary = src.read(1, masked=True)
    im = ax_p.imshow(prob, cmap="viridis", vmin=0, vmax=1)
    ax_p.set_title("a) Old-growth probability", loc="left")
    fig.colorbar(im, ax=ax_p, shrink=0.7, label="p(old-growth)")
    ax_b.imshow(binary, cmap="Greens", vmin=0, vmax=1)
    ax_b.set_title("b) Old-growth (F1-max threshold)", loc="left")
    for ax in (ax_p, ax_b):
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle("Wall-to-wall old-growth map (XGBoost + TESSERA)", fontsize=11)
    plt.show()  # display inline only; no PDF saved for the wall-to-wall map
else:
    print("Wall-to-wall GeoTIFFs not found; run scripts/run_final_inference.py.")

## Calibration

- The **parcel** calibrator recalibrates the parcel-mean probability stored in the vector GeoPackage
- The **pixel** calibrator recalibrates the wall-to-wall probability raster.

Although five calibrators are compared, **Platt** scaling is used for the final maps.

In [ ]:
def _comparison_bar(ax, table, selected, title):
    """ECE bar chart for one calibration level, highlighting the selected method."""
    methods = [m for m in table["method"] if m != "uncalibrated"]
    ax.bar(
        np.arange(len(methods)),
        table.set_index("method").loc[methods, "ece"],
        color=[PALETTE_CATEGORICAL["blue"] if m == selected else "0.7" for m in methods],
    )
    ax.set_xticks(np.arange(len(methods)))
    ax.set_xticklabels(methods, rotation=20)
    ax.set_ylabel("Cross-validated ECE")
    ax.set_title(title, loc="left")
    ax.spines[["top", "right"]].set_visible(False)


levels = [
    ("parcel", calibration, metadata.get("selected_calibrator") if metadata else None),
    ("pixel", calibration_pixel, metadata.get("selected_calibrator_pixel") if metadata else None),
]
available = [(lvl, tbl, sel) for lvl, tbl, sel in levels if tbl is not None]
if available:
    for lvl, tbl, sel in available:
        show = tbl.copy()
        show["selected"] = show["method"] == sel
        print(f"[Calibration - {lvl}] selected method: {sel}")
        print(show.round(4).to_string(index=False))
        print()

    fig, axes = plt.subplots(
        1, len(available), figsize=get_figure_size("double", aspect=0.42), constrained_layout=True
    )
    axes = np.atleast_1d(axes)
    for ax, (lvl, tbl, sel), letter in zip(axes, available, "ab", strict=False):
        _comparison_bar(ax, tbl, sel, f"{letter}) {lvl}-level (selected highlighted)")
    fig.suptitle("Calibrator comparison by ECE (lower is better)", fontsize=11)
    save_figure(
        fig,
        f"{NOTEBOOK}/calibration_comparison",
        data={lvl: tbl for lvl, tbl, _ in available},
    )
else:
    print("calibration_comparison[_pixel].csv not found; run scripts/run_final_inference.py.")

In [ ]:
from sklearn.model_selection import StratifiedKFold

from utils.calibration import CALIBRATION_METHODS, fit_calibrator
from utils.terminology import SEED

# Reliability diagrams, one panel per calibrator plus the uncalibrated scores, for each
# calibration level. The bar chart ranks the calibrators by a single number (ECE); these curves
# show *where* on the probability range each transform is well or poorly calibrated and how sharp
# (confident) it is.
_N_BINS = 10  # equal-width bins, matching the ECE definition in utils.calibration
_Z = 1.959963984540054  # 95% normal quantile for the Wilson interval


def _cv_calibrated(y, p, method, n_splits=5, seed=SEED):
    """Out-of-fold calibrated probabilities, using the same CV scheme as the comparison table.

    Refitting the calibrator on each training split and scoring the held-out split keeps the
    flexible non-parametric methods (isotonic, spline) honest: their reliability is judged on
    samples they were not fitted on, so the curves correspond to the cross-validated ECE shown in
    the bar chart rather than an optimistic in-sample fit.
    """
    if method == "uncalibrated":
        return np.asarray(p, dtype=float)
    out = np.empty(p.shape[0], dtype=float)
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for train_idx, val_idx in splitter.split(p, y):
        cal = fit_calibrator(method, y[train_idx], p[train_idx])
        out[val_idx] = cal.predict(p[val_idx])
    return out


def _reliability(y, p, n_bins=_N_BINS):
    """Per-bin mean predicted probability, observed frequency and a Wilson 95% interval."""
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        mask = idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        frac = float(y[mask].mean())
        denom = 1.0 + _Z**2 / n
        centre = (frac + _Z**2 / (2 * n)) / denom
        half = _Z * np.sqrt(frac * (1 - frac) / n + _Z**2 / (4 * n**2)) / denom
        rows.append(
            {
                "bin": b,
                "mean_pred": float(p[mask].mean()),
                "obs_freq": frac,
                "ci_lo": max(0.0, centre - half),
                "ci_hi": min(1.0, centre + half),
                "count": n,
            }
        )
    return pd.DataFrame(rows)


def reliability_panels(y, p_raw, comparison, selected, suptitle, fig_name):
    """A panel of cross-validated reliability diagrams (uncalibrated + each calibrator)."""
    panels = ["uncalibrated", *CALIBRATION_METHODS]
    ece_by_method = comparison.set_index("method")["ece"].to_dict()
    brier_by_method = comparison.set_index("method")["brier"].to_dict()

    ncol = 3
    nrow = int(np.ceil(len(panels) / ncol))
    fig, axes = plt.subplots(
        nrow,
        ncol,
        figsize=get_figure_size("double", aspect=0.72),
        constrained_layout=True,
        sharex=True,
        sharey=True,
    )
    axes = np.atleast_1d(axes).ravel()
    curve_data = {}
    for ax, method, letter in zip(axes, panels, "abcdefgh", strict=False):
        p_cal = _cv_calibrated(y, p_raw, method)
        curve = _reliability(y, p_cal)
        curve_data[method] = curve
        is_sel = method == selected
        colour = PALETTE_CATEGORICAL["blue"] if is_sel else "0.35"

        # Score distribution underlay (sharpness): a calibrated but flat predictor is useless.
        hist_ax = ax.twinx()
        hist_ax.hist(p_cal, bins=np.linspace(0, 1, _N_BINS + 1), color="0.8", alpha=0.6, zorder=0)
        hist_ax.set_yticks([])
        hist_ax.set_ylim(0, None)

        ax.plot([0, 1], [0, 1], ls="--", lw=0.8, color="0.6", zorder=1)
        ax.errorbar(
            curve["mean_pred"],
            curve["obs_freq"],
            yerr=[curve["obs_freq"] - curve["ci_lo"], curve["ci_hi"] - curve["obs_freq"]],
            fmt="o-",
            ms=3.5,
            lw=1.2,
            color=colour,
            ecolor=colour,
            elinewidth=0.8,
            capsize=2,
            zorder=3,
        )
        ax.set_zorder(hist_ax.get_zorder() + 1)
        ax.patch.set_visible(False)
        ax.set_title(
            f"{letter}) {method}" + (" (selected)" if is_sel else ""),
            loc="left",
            color=colour,
            fontweight="bold" if is_sel else "normal",
        )
        ece, brier = ece_by_method.get(method, np.nan), brier_by_method.get(method, np.nan)
        ax.text(
            0.04,
            0.96,
            f"ECE {ece:.3f}\nBrier {brier:.3f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=7,
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.spines[["top", "right"]].set_visible(False)

    for ax in axes[len(panels) :]:
        ax.set_visible(False)
    for ax in axes[: len(panels)]:
        if ax.get_subplotspec().is_last_row():
            ax.set_xlabel("Mean predicted probability")
        if ax.get_subplotspec().is_first_col():
            ax.set_ylabel("Observed frequency")
    fig.suptitle(suptitle, fontsize=11)
    save_figure(fig, fig_name, data=curve_data)


_reliability_levels = [
    (
        "parcel",
        oof_parcels,
        "p_mean",
        calibration,
        metadata.get("selected_calibrator") if metadata else None,
    ),
    (
        "pixel",
        oof_pixels,
        "p",
        calibration_pixel,
        metadata.get("selected_calibrator_pixel") if metadata else None,
    ),
]
for lvl, oof, pcol, comparison, selected in _reliability_levels:
    if oof is None or comparison is None:
        print(f"OOF or comparison for {lvl} level not found; run scripts/run_final_inference.py.")
        continue
    reliability_panels(
        oof["y_true"].to_numpy().astype(int),
        oof[pcol].to_numpy().astype(float),
        comparison,
        selected,
        f"Reliability of each {lvl}-level calibrator (cross-validated; grey = score distribution)",
        f"{NOTEBOOK}/calibration_reliability_{lvl}",
    )

## Predicted old-growth area and its uncertainty

The predicted area is the total area of the parcels this study classifies as old-growth (parcel
mean probability at or above the parcel F1-max threshold, the operating point used by the parcel
product and the cross-study comparison in notebook 011).

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter


def _ci(values, level):
    lo, hi = np.percentile(values, [(100 - level) / 2, 100 - (100 - level) / 2])
    return lo, hi


def _row_major(handles, labels, ncol):
    """Reorder handles so a (column-major) matplotlib legend reads left-to-right, top-to-bottom."""
    n = len(handles)
    nrow = int(np.ceil(n / ncol))
    order = [r * ncol + c for c in range(ncol) for r in range(nrow) if r * ncol + c < n]
    return [handles[i] for i in order], [labels[i] for i in order]


if area_boot is not None and len(area_boot) > 0 and "parcel_area_ha" in area_boot.columns:
    # Parcel-consistent area per replicate (pixels in parcels whose mean probability clears the
    # parcel F1-max threshold). Platt calibration is monotonic, so this is the Platt parcel-verdict
    # area; the CSV also carries pixel_area_ha if the per-pixel operating point is wanted instead.
    areas = area_boot["parcel_area_ha"].to_numpy()
    boot_mean = float(areas.mean())
    point_est = metadata.get("predicted_ogf_area_parcel_ha") if metadata else None
    ci_lo, ci_hi = _ci(areas, 95)
    table = pd.DataFrame(
        [
            {"interval": f"{level}%", "lo_ha": _ci(areas, level)[0], "hi_ha": _ci(areas, level)[1]}
            for level in (90, 95, 99)
        ]
    )
    print(f"[Area] {len(area_boot)} bootstrap replicates; bootstrap mean {boot_mean:,.0f} ha")
    if point_est is not None:
        print(f"[Area] deployed point estimate: {point_est:,.0f} ha")
    print(table.round(0).to_string(index=False))

    # Contrasting (non-green) colours so the lines stand out against the light-green bars.
    product_colours = {
        "sabatini": PALETTE_CATEGORICAL["orange"],
        "munteanu": PALETTE_CATEGORICAL["blue"],
        "kathmann": PALETTE_CATEGORICAL["magenta"],
        "schickhofer": PALETTE_CATEGORICAL["teal"],
    }
    fig, ax = plt.subplots(figsize=get_figure_size("double", aspect=0.45), constrained_layout=True)
    ax.hist(
        areas,
        bins=60,
        color=PALETTE_CATEGORICAL["light_green"],
        alpha=0.55,
        edgecolor="0.35",
        linewidth=0.4,
    )
    ax.axvspan(ci_lo, ci_hi, color="0.6", alpha=0.2, lw=0, zorder=0)

    # Legend order: this study's deployed point estimate (solid black) is the area of the published
    # map; the block-retraining bootstrap mean (dashed black) averages the resample-retrained
    # replicates and sits slightly lower; the grey band is the 95% bootstrap interval. Then the four
    # existing products' parcel-verdict areas (solid, same colours).
    handles, labels = [], []
    if point_est is not None:
        handles.append(ax.axvline(point_est, color="black", ls="-", lw=1.5, zorder=4))
        labels.append(f"Point estimate ({point_est:,.0f} ha)")
    handles.append(ax.axvline(boot_mean, color="black", ls="--", lw=1.5, zorder=4))
    labels.append(f"Bootstrap mean ({boot_mean:,.0f} ha)")
    handles.append(mpatches.Patch(facecolor="0.6", alpha=0.2, edgecolor="none"))
    labels.append(f"95% CI ({ci_lo:,.0f}-{ci_hi:,.0f} ha)")

    # Existing-product reference lines (solid, same colours): each product's PARCEL-verdict area
    # (summed geometry of the whole parcels it calls old-growth) -- the same parcel basis as this
    # study's bootstrap distribution, matching nb 011's parcel column. The published GeoPackage
    # carries only this study's outputs, so the product verdicts are read from the comparison run
    # it was packaged from (recorded in final_outputs_metadata.json).
    final_meta = paths.results / "final" / "final_outputs_metadata.json"
    if final_meta.is_file():
        import geopandas as gpd

        comparison_run = json.loads(final_meta.read_text())["comparison_run"]
        gpar = gpd.read_file(
            paths.results / "comparison" / comparison_run / "product_parcel_comparison.gpkg"
        )
        gpar["parcel_area_ha"] = gpar.to_crs(3035).geometry.area / 1e4
        for study in ("sabatini", "munteanu", "kathmann", "schickhofer"):
            verdict = gpar[f"ogf_{study}"].astype(bool)
            area = float(gpar.loc[verdict, "parcel_area_ha"].sum())
            author = COMPARISON_STUDIES[study].display.split(" (")[0]  # e.g. "Sabatini"
            handles.append(
                ax.axvline(area, color=product_colours.get(study, "0.5"), ls="-", lw=1.5, zorder=3)
            )
            labels.append(f"{author} ({area:,.0f} ha)")

    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.set_xlabel("Old-growth parcel area (ha)")
    ax.set_ylabel(f"Bootstrap replicates (n={len(area_boot)})")
    ax.set_title("Predicted old-growth parcel area", loc="left")
    ax.spines[["top", "right"]].set_visible(False)

    # One legend below the axes, wrapped over two rows and reordered so it reads left-to-right,
    # top-to-bottom (matplotlib fills legend columns top-to-bottom by default).
    h, la = _row_major(handles, labels, ncol=4)
    fig.legend(h, la, loc="outside lower center", ncol=4, frameon=False, fontsize=7)
    save_figure(fig, f"{NOTEBOOK}/fig_s8_area_bootstrap", data=area_boot)
else:
    print("area_bootstrap.csv missing/incomplete; (re-)run scripts/run_area_bootstrap.py.")

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize

from utils import vector_io

# Presentation tables for the deployed old-growth parcel map (XGBoost + TESSERA), with the parcel
# point estimate as the single reference. Two tables give every area as a percentage of the full
# AOI and of the forested AOI (CORINE forest within the Natura 2000 boundary). Extent carries the
# block-retraining bootstrap 95% CI; parcel count, the CORINE forest-type breakdown (area
# apportioned to the reference, with parcel count) and the FABDEM altitude (mean and median) are
# point estimates from the deployed verdict.
_gpkg = paths.results / "final" / "ogf_labels_predictions.gpkg"
_elev = paths.processed / "rasters" / "fabdem_10m" / "elevation_3035_10m.tif"
_corine = paths.processed / "vectors" / "corine_land_cover" / "corine_forest_aoi_3035.gpkg"
if area_boot is None or metadata is None or "parcel_area_ha" not in area_boot.columns:
    print("Area bootstrap / metadata unavailable; run the inference and area-bootstrap scripts.")
elif not (_gpkg.is_file() and _elev.is_file() and _corine.is_file() and paths.aoi.is_file()):
    print("Final gpkg, FABDEM elevation, CORINE forest or AOI polygon not found; run the pipeline.")
else:
    aoi_ha = vector_io.unique_area_ha(gpd.read_file(paths.aoi))  # full study-area denominator
    forest_ha = float(gpd.read_file(_corine).geometry.area.sum() / 1e4)  # CORINE forest within AOI
    domain_ha = metadata.get("n_mapped_pixels", 0) * 0.01
    parcel_point = metadata.get("predicted_ogf_area_parcel_ha")  # the single reference
    pixel_point = metadata.get("predicted_ogf_area_ha")
    p_lo, p_hi = np.percentile(area_boot["parcel_area_ha"].to_numpy(), [2.5, 97.5])
    px_lo, px_hi = np.percentile(area_boot["pixel_area_ha"].to_numpy(), [2.5, 97.5])
    parcel_boot = float(area_boot["parcel_area_ha"].mean())
    pixel_boot = float(area_boot["pixel_area_ha"].mean())
    na = "—"

    parcels = gpd.read_file(_gpkg, layer="ogf_labels_predictions").to_crs(3035)
    # OGF_binary is null for the parcels without a prediction (no pixel centre) and reads
    # back as NaN, which astype(bool) would count as True; compare with 1 instead.
    ogf = parcels[parcels["OGF_binary"] == 1].copy()
    ogf["poly_ha"] = ogf.geometry.area / 1e4
    ogf["forest_type"] = ogf["CORINE_forest_type"].fillna("unclassified")
    types = ["broadleaf", "coniferous", "mixed", "unclassified"]
    code = {t: i + 1 for i, t in enumerate(types)}
    ogf["code"] = ogf["forest_type"].map(code).fillna(0).astype("uint8")

    # Altitude: rasterise the type-coded OGF parcels onto the FABDEM grid, then take mean and median
    # elevation within each type's pixels (Dropbox raster reads are occasionally flaky, so retry).
    _last = None
    for _attempt in range(3):
        try:
            with rasterio.open(_elev) as _src:
                _elevation = _src.read(1, masked=True)
                _grp = rasterize(
                    zip(ogf.geometry, ogf["code"], strict=True),
                    out_shape=(_src.height, _src.width),
                    transform=_src.transform,
                    fill=0,
                    dtype="uint8",
                )
            break
        except Exception as _exc:
            _last = _exc
    else:
        raise _last
    _ev = np.asarray(_elevation)
    _valid = (~np.ma.getmaskarray(_elevation)) & (_grp > 0)
    mean_alt, med_alt = {}, {}
    for t in types:
        _px = _ev[_valid & (_grp == code[t])]
        mean_alt[t] = _px.mean() if _px.size else np.nan
        med_alt[t] = np.median(_px) if _px.size else np.nan
    mean_all, med_all = _ev[_valid].mean(), np.median(_ev[_valid])

    area_by = ogf.groupby("forest_type")["poly_ha"].sum()
    n_by = ogf.groupby("forest_type").size()
    scale = parcel_point / area_by.sum()  # apportion polygon areas to the parcel point estimate
    ha_by = {t: area_by.get(t, 0.0) * scale for t in types}

    def _build(denom):
        def _cell(t):
            return f"{ha_by[t]:,.0f} ha ({n_by.get(t, 0):,}; {ha_by[t] / denom * 100:.1f}%)"

        return pd.DataFrame(
            [
                {
                    "statistic": "point estimate",
                    "extent": f"{parcel_point:,.0f} ha ({parcel_point / denom * 100:.1f}%)",
                    "parcels": f"{len(ogf):,}",
                    "broadleaf": _cell("broadleaf"),
                    "coniferous": _cell("coniferous"),
                    "mixed": _cell("mixed"),
                    "unclassified": _cell("unclassified"),
                    "mean alt (m)": f"{mean_all:,.0f}",
                    "median alt (m)": f"{med_all:,.0f}",
                },
                {
                    "statistic": "95% CI",
                    "extent": (
                        f"{p_lo:,.0f}-{p_hi:,.0f} ha "
                        f"({p_lo / denom * 100:.1f}-{p_hi / denom * 100:.1f}%)"
                    ),
                    "parcels": na,
                    "broadleaf": na,
                    "coniferous": na,
                    "mixed": na,
                    "unclassified": na,
                    "mean alt (m)": na,
                    "median alt (m)": na,
                },
            ]
        )

    print(
        f"Predicted old-growth — deployed parcel map (XGBoost + TESSERA); "
        f"parcel domain {domain_ha:,.0f} ha."
    )
    print(f"\nAs % of AOI ({aoi_ha:,.0f} ha):")
    print(_build(aoi_ha).to_string(index=False))
    print(
        f"\nAs % of forested AOI ({forest_ha:,.0f} ha; CORINE forest within the AOI, "
        f"{forest_ha / aoi_ha * 100:.1f}% of AOI):"
    )
    print(_build(forest_ha).to_string(index=False))
    print("\nCORINE columns show area (parcel count; %), apportioned to the parcel point estimate.")
    print(
        "Altitude by type (mean/median m): "
        + ", ".join(f"{t} {mean_alt[t]:,.0f}/{med_alt[t]:,.0f}" for t in types)
        + "."
    )
    px_aoi, px_for = pixel_point / aoi_ha * 100, pixel_point / forest_ha * 100
    print(
        f"Parcel bootstrap mean {parcel_boot:,.0f} ha. "
        f"Pixel-level point estimate {pixel_point:,.0f} ha "
        f"({px_aoi:.1f}% AOI / {px_for:.1f}% forest; bootstrap mean {pixel_boot:,.0f} ha; "
        f"95% CI {px_lo:,.0f}-{px_hi:,.0f} ha)."
    )